In [ ]:
from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel



In [ ]:
load_dotenv()

groq_model = ChatGroq(model_name = "llama-3.3-70b-versatile")

llama_model = ChatOllama(
    model = "llama3.2",
    base_url="http://localhost:11434",
    temperature=0.8,
    )




In [ ]:
prompt1 = PromptTemplate(template="Explain about {topic} in detail",
                        input_variables=["topic"])

prompt2 = PromptTemplate(template="Genarate notes from this {text} ",
                        input_variables=["text"])

prompt3 = PromptTemplate(template="Generates five quiz like qustions from this {text}",
                        input_variables=["text"])

prompt4 = PromptTemplate(template="Merge these two notes and quiz {notes} {quiz}",
                        input_variables=["notes","quiz"])



In [ ]:
parser = StrOutputParser()

In [ ]:
parallel_chain = RunnableParallel({
    "notes" : prompt2 | groq_model | parser,
    "quiz" : prompt3 | groq_model | parser
})

merge_chain = prompt4 | groq_model | parser

chain = prompt1 | llama_model | parser | parallel_chain | merge_chain 

In [ ]:
def chatbot(asks):
    result = chain.invoke({asks})
    return result

In [ ]:
chain.get_graph().print_ascii()

In [ ]:
# UI Creation

import gradio as gr


with gr.Blocks() as app:
    
    gr.Markdown("# StudySnap")
    gr.Markdown("### Learn anything. Instantly. Visually.")
    gr.Markdown("Drop a topic below and let's break it down ")

    asks = gr.Textbox(
        placeholder="e.g. Neural Networks, Photosynthesis, Python Loops...",
        label=" What do you want to explore?"
    )

    submit = gr.Button("⚡ Snap It")

    # Output 1
    gr.Markdown("## Explanation")
    text_output = gr.Textbox(label="Your concept, simplified")

    # KEY FIX HERE
    submit.click(
        fn=chatbot,
        inputs=asks,
        outputs=text_output
    )

    clear = gr.ClearButton([asks, text_output])

app.launch(share=True)

In [ ]:
# # UI
# import gradio as gr

# with gr.Blocks() as app:
#     gr.Markdown("# StudySnap")
#     gr.Markdown("Welcome Geeks. What do you want to learn today?  ")

#     asks = gr.Textbox(placeholder = "Enter your Topic Here : ")
#     output = gr.Textbox(label= "response")
#     submit = gr.Button("Ask")
#     submit.click(fn=chatbot, inputs=asks, outputs=output)
#     clear = gr.ClearButton([asks, output])
    
# app.launch()

In [ ]:
# # UI
# import gradio as gr

# with gr.Blocks() as app:
#     gr.Markdown("Displaying Chain Graph")
#     output = gr.Textbox(label= "response")
#     submit = gr.Button("Ask")
#     submit.click(fn=graph, outputs=output)
#     clear = gr.ClearButton([asks, output])
    
# app.launch()